<div style="background-color:#e6f2ff; padding:20px; border-radius:10px;">
<img style="float:left; margin-right:20px;" src='Figures/alinco.png' width="120"/>
<h1 style="color:#000047;">Actividad 5:  Redes Neuronales Recurrentes</h1>
<br style="clear:both"/>
</div>

<div style="border-left:4px solid #000047; padding:10px; margin-top:10px; background:#f5f5f5;">
<b>Objetivo:</b> Esta actividad consiste en obtener un modelo RNN para los datos "international-airline-passengers.csv". Este dataset contiene un informe del Aeropuerto Internacional de San Francisco sobre estadísticas mensuales de tráfico de pasajeros por aerolínea. Los datos del aeropuerto son estacionales por naturaleza, por lo tanto, cualquier análisis comparativo debe realizarse de un período a otro (es decir, enero de 2010 frente a enero de 2009) y no de un período a otro (es decir, enero de 2010 frente a febrero de 2010). .
</div>


## 1. Modelo de red neuronal simple

### Cargar los datos

In [1]:
import pandas as pd
import numpy as np

In [2]:
dataset_train = pd.read_csv('Data/international-airline-passengers.csv')

In [3]:
dataset_train.columns

Index(['Month', 'International airline passengers: monthly totals in thousands. Jan 49 ? Dec 60'], dtype='object')

### Explorar los datos

In [4]:
dataset_train.info

<bound method DataFrame.info of                                                  Month  \
0                                              1949-01   
1                                              1949-02   
2                                              1949-03   
3                                              1949-04   
4                                              1949-05   
..                                                 ...   
140                                            1960-09   
141                                            1960-10   
142                                            1960-11   
143                                            1960-12   
144  International airline passengers: monthly tota...   

     International airline passengers: monthly totals in thousands. Jan 49 ? Dec 60  
0                                                112.0                               
1                                                118.0                               
2            

### Obtener los datos para el entrenamiento

In [5]:
train = dataset_train.loc[:, ['International airline passengers: monthly totals in thousands. Jan 49 ? Dec 60']].values
train

array([[112.],
       [118.],
       [132.],
       [129.],
       [121.],
       [135.],
       [148.],
       [148.],
       [136.],
       [119.],
       [104.],
       [118.],
       [115.],
       [126.],
       [141.],
       [135.],
       [125.],
       [149.],
       [170.],
       [170.],
       [158.],
       [133.],
       [114.],
       [140.],
       [145.],
       [150.],
       [178.],
       [163.],
       [172.],
       [178.],
       [199.],
       [199.],
       [184.],
       [162.],
       [146.],
       [166.],
       [171.],
       [180.],
       [193.],
       [181.],
       [183.],
       [218.],
       [230.],
       [242.],
       [209.],
       [191.],
       [172.],
       [194.],
       [196.],
       [196.],
       [236.],
       [235.],
       [229.],
       [243.],
       [264.],
       [272.],
       [237.],
       [211.],
       [180.],
       [201.],
       [204.],
       [188.],
       [235.],
       [227.],
       [234.],
       [264.],
       [30

### Preprocemaniento de datos

In [6]:
# escalamiento de los datos (MinMaxScaler)
from sklearn.preprocessing import MinMaxScaler

In [7]:
scaler = MinMaxScaler(feature_range=(0,1))
train_scaler = scaler.fit_transform(train)
train_scaler

array([[0.01544402],
       [0.02702703],
       [0.05405405],
       [0.04826255],
       [0.03281853],
       [0.05984556],
       [0.08494208],
       [0.08494208],
       [0.06177606],
       [0.02895753],
       [0.        ],
       [0.02702703],
       [0.02123552],
       [0.04247104],
       [0.07142857],
       [0.05984556],
       [0.04054054],
       [0.08687259],
       [0.12741313],
       [0.12741313],
       [0.1042471 ],
       [0.05598456],
       [0.01930502],
       [0.06949807],
       [0.07915058],
       [0.08880309],
       [0.14285714],
       [0.11389961],
       [0.13127413],
       [0.14285714],
       [0.18339768],
       [0.18339768],
       [0.15444015],
       [0.11196911],
       [0.08108108],
       [0.11969112],
       [0.12934363],
       [0.14671815],
       [0.17181467],
       [0.14864865],
       [0.15250965],
       [0.22007722],
       [0.24324324],
       [0.26640927],
       [0.2027027 ],
       [0.16795367],
       [0.13127413],
       [0.173

### Crear la estructura de datos

In [10]:
SEQ_LENGTH = 50

def create_sliding_windows(data, seq_length):
    X, y = [], []
    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i, 0])
        y.append(data[i,0])
        
    return np.array(X), np.array(y)

In [12]:
X_train, y_train = create_sliding_windows(train_scaler, SEQ_LENGTH)


In [13]:
X_train

array([[0.01544402, 0.02702703, 0.05405405, ..., 0.17374517, 0.17760618,
        0.17760618],
       [0.02702703, 0.05405405, 0.04826255, ..., 0.17760618, 0.17760618,
        0.25482625],
       [0.05405405, 0.04826255, 0.03281853, ..., 0.17760618, 0.25482625,
        0.25289575],
       ...,
       [0.48455598, 0.38996139, 0.32239382, ..., 0.96911197, 0.77992278,
        0.68918919],
       [0.38996139, 0.32239382, 0.38996139, ..., 0.77992278, 0.68918919,
        0.55212355],
       [0.32239382, 0.38996139, 0.40733591, ..., 0.68918919, 0.55212355,
        0.63320463]], shape=(95, 50))

In [14]:
#Reshaping:
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [15]:
X_train.shape

(95, 50, 1)

### Crear un modelo de red neuronal

<div class="alert alert-success">
  <strong>Task:</strong> Crear una RNN simple.
<ul>
  <li>capa SimpleRnn(units=50, act=tanh) -> dropout(0.2) -> SimpleRNN(units=50, act=tanh)-> dropout(0.2)->dense(units=1)</li>
  <li>Muestre un resumen de la red</li>
  <li>Compilar el modelo con `Adam` como el optimizador</li>
</ul>

</div>

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Dropout

In [23]:
reg = Sequential()

#Agregar las capas
reg.add(SimpleRNN(units = 50, activation = 'tanh', return_sequences = True, input_shape= (X_train.shape[1],1)))
reg.add(Dropout(0.2))

reg.add(SimpleRNN(units = 50, activation = 'tanh', return_sequences = True,))
reg.add(Dropout(0.2))

reg.add(Dense(units=1))

reg.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn_5 (SimpleRNN)             │ (None, 50, 50)              │           2,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 50, 50)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ simple_rnn_6 (SimpleRNN)             │ (None, 50, 50)              │           5,050 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 50, 50)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 50, 1)               │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,701 (30.08 KB)

 Trainable params: 7,701 (30.08 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
reg.compile(optimizer = 'adam', loss = 'mean_squared_error')

### Entrenar el modelo


In [25]:
reg.fit(X_train, y_train, epochs=100, batch_size=32)

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: nan 
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: nan
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: nan
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: nan
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: nan
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: nan
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: nan
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: nan
Epoch 9/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: nan
Epoch 10/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: nan
Epoch 11/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: nan
Epoch 12/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: nan
Epoch 13/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: nan
Epoch 14/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: nan
Epoch 15/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: nan
Epoch 16/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: nan


### Predicción y visualización del modelo RNN


## 2.- Modelo de una red recurrente con LSTM

### Cargar datos

In [ ]:
data = pd.read_csv('Data/international-airline-passengers.csv')
data.head()

### Preprocesamiento de datos

### Crear la estructura de datos

### Crear un modelo de redes neuronales recurrentes LSTM

<div class="alert alert-success">
  <strong>Task:</strong> Crear una RNN simple.
<ul>
  <li>capa LSTM(units=10, act=tanh) -> dense(units=1)</li>
  <li>Muestre un resumen de la red</li>
  <li>Compilar el modelo con `Adam` como el optimizador loss=`mean_squared_error`</li>
</ul>

</div>

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

In [ ]:
# modelo


### Predicción

In [ ]:
#make predictions


In [ ]:
# shifting train
